# Entraînement local — test avant push Kaggle

Notebook miroir de `train_kaggle.ipynb` pour valider le code en local.

**Différences vs Kaggle :**
- Chemins locaux (pas de `git clone`, pas de `pip install`)
- Section V3 multi-GPU **ignorée** (pas de GPU)
- `n_epochs` réduits à 5 via monkey-patch pour aller vite

## 0. Setup

In [1]:
import sys
from pathlib import Path

# ⚙️  CONFIGURATION LOCALE
DATA_CSV_DEST = Path("/home/vm2lucas/Bureau/Hackathon/hackathon2026/.claude/worktrees/repro-vieux-briscards/external/vieux_briscards/data/segment_alerts_all_airports_train.csv")
REPO_DIR      = Path("/home/vm2lucas/Bureau/Hackathon/prediction-orage")
OUTPUT_DIR    = REPO_DIR / "outputs_local"

(REPO_DIR / "data").mkdir(exist_ok=True)
(REPO_DIR / "models").mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

FEATURES_PARQUET = REPO_DIR / "data" / "features_survival.parquet"

SRC_DIR = str(REPO_DIR / "src")
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

print(f"Repo    : {REPO_DIR}")
print(f"CSV     : {DATA_CSV_DEST} (exists={DATA_CSV_DEST.exists()})")
print(f"Outputs : {OUTPUT_DIR}")

Repo    : /home/vm2lucas/Bureau/Hackathon/prediction-orage
CSV     : /home/vm2lucas/Bureau/Hackathon/hackathon2026/.claude/worktrees/repro-vieux-briscards/external/vieux_briscards/data/segment_alerts_all_airports_train.csv (exists=True)
Outputs : /home/vm2lucas/Bureau/Hackathon/prediction-orage/outputs_local


In [2]:
# Monkey-patch : réduit n_epochs à 5 pour tous les entraînements
# (smoke test rapide sur CPU ; retirer cette cellule pour un vrai run)

N_EPOCHS_LOCAL = 5

def _make_fast_fit(original_fit):
    def fast_fit(self, *args, **kwargs):
        kwargs["n_epochs"] = N_EPOCHS_LOCAL
        return original_fit(self, *args, **kwargs)
    return fast_fit

# Appliqué après chaque import pour ne pas dépendre de l'ordre
print(f"Monkey-patch actif : n_epochs={N_EPOCHS_LOCAL} pour tous les .fit()")

Monkey-patch actif : n_epochs=5 pour tous les .fit()


## 1. Chargement des données et features de survie

In [10]:
import data_loader
import features as feat_module

print("Chargement du CSV...")
df     = data_loader.load_raw(DATA_CSV_DEST)
alerts = data_loader.load_alerts(df)
print(f"  {len(df):,} éclairs chargés, {len(alerts):,} en session d'alerte")

if not FEATURES_PARQUET.exists():
    print("Construction des features de survie (XGBoost / BNN)...")
    feat_df = feat_module.build_features(alerts)
    feat_df.to_parquet(FEATURES_PARQUET, index=False)
    print(f"  {len(feat_df):,} lignes sauvegardées → {FEATURES_PARQUET}")
else:
    print(f"  Features déjà présentes : {FEATURES_PARQUET}")

Chargement du CSV...
  507,071 éclairs chargés, 220,498 en session d'alerte
Construction des features de survie (XGBoost / BNN)...


Feature engineering par session: 100%|██████████| 3321/3321 [01:41<00:00, 32.71it/s]
/home/vm2lucas/Bureau/Hackathon/prediction-orage/src/features.py:116: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  res[num_cols] = res[num_cols].fillna(0)


  215,559 lignes sauvegardées → /home/vm2lucas/Bureau/Hackathon/prediction-orage/data/features_survival.parquet


## 2. Phase 1 — Baseline 30 min & XGBoost survival:AFT

In [ ]:
import evaluation

print("Chargement des features de survie...")
feat_df = evaluation.load_features()

print("Split temporel...")
train_tab, val_tab = evaluation.temporal_split(feat_df)

val_tab = evaluation.baseline_30min(val_tab)
baseline_metrics = evaluation.compute_metrics(val_tab, "pred_baseline", "Baseline 30 min")

xgb_model, val_tab = evaluation.train_xgboost_survival(train_tab, val_tab)
xgb_metrics = evaluation.compute_metrics(val_tab, "pred_xgb_aft", "XGBoost survival:AFT")

xgb_model.save_model(str(REPO_DIR / "models" / "xgboost_aft.json"))
print("Modèle sauvegardé : models/xgboost_aft.json")

## 3. Phase 1 — BNN (Monte Carlo Dropout sur MLP)

In [ ]:
import bnn_model

print("Chargement et split...")
train_bnn, val_bnn = bnn_model.load_and_split()
print(f"  Train : {len(train_bnn):,} | Val : {len(val_bnn):,}")

bnn, scaler, y_pred_bnn, y_std_bnn, y_true_bnn = bnn_model.train_bnn(
    train_bnn, val_bnn, n_epochs=N_EPOCHS_LOCAL
)
print("Modèle sauvegardé : models/bnn_mc_dropout.pt")

## 4. Phase 2 — Hawkes Classique & Neural Hawkes GRU V1

In [ ]:
import hawkes_models

# Monkey-patch NeuralHawkes.fit
hawkes_models.NeuralHawkes.fit = _make_fast_fit(hawkes_models.NeuralHawkes.fit)

print("Préparation des sessions Hawkes (4 features)...")
sessions_hawkes = hawkes_models.prepare_hawkes_sessions(alerts)
print(f"  {len(sessions_hawkes)} sessions")

hawkes_classique, neural_hawkes_v1 = hawkes_models.evaluate_hawkes_models(sessions_hawkes)

import json, torch
params = {"mu": hawkes_classique.mu, "alpha": hawkes_classique.alpha, "beta": hawkes_classique.beta}
with open(str(REPO_DIR / "models" / "hawkes_classique_params.json"), "w") as f:
    json.dump(params, f, indent=2)
torch.save(neural_hawkes_v1.model.state_dict(), str(REPO_DIR / "models" / "neural_hawkes_gru_v1.pt"))
print("Modèles sauvegardés")

## 5. Phase 2bis — Neural Hawkes V2 (multi-tâche, 12 features)

In [ ]:
import neural_hawkes_v2

# Monkey-patch NeuralHawkesTrainer.fit
neural_hawkes_v2.NeuralHawkesTrainer.fit = _make_fast_fit(neural_hawkes_v2.NeuralHawkesTrainer.fit)

print("Préparation des sessions V2 (12 features)...")
sessions_v2 = neural_hawkes_v2.prepare_sessions_v2(alerts)
print(f"  {len(sessions_v2)} sessions, dim={sessions_v2[0]['features'].shape[1]}")

results_v2, trainer_tf_v2, trainer_tf_lg_v2 = neural_hawkes_v2.evaluate_all_variants(sessions_v2)

print("\nRésultats V2 :")
for key, r in results_v2.items():
    print(f"  {key:35s} | MAE={r['mae']:.2f} | Biais={r['bias']:+.2f}")

## 6. Phase 2bis — Neural Hawkes V3 Gaussien (multi-GPU) — ⏭ ignoré en local

Cette section utilise `train_parallel.run_v3()` qui dispatche sur les GPU.  
Sur Kaggle T4 x2 elle tourne normalement. En local (pas de GPU), on la saute.

In [ ]:
print("Section V3 multi-GPU ignorée en local.")
results_v3 = {}  # dict vide pour que la cellule résumé ne plante pas

## 7. Phase 2bis — Hawkes Bayésien (MC Dropout + Variationnel)

In [ ]:
import bayesian_hawkes

# Monkey-patch les deux trainers bayésiens
bayesian_hawkes.BayesianHawkesTrainer.fit = _make_fast_fit(bayesian_hawkes.BayesianHawkesTrainer.fit)
bayesian_hawkes.VariationalHawkesTrainer.fit = _make_fast_fit(bayesian_hawkes.VariationalHawkesTrainer.fit)

# sessions_v2 déjà préparées à la cellule 5
results_bayes, trainer_mc, trainer_var = bayesian_hawkes.run_bayesian_experiments(sessions_v2)

print("\nRésultats Bayésiens :")
for key, r in results_bayes.items():
    cal = f"{r.get('calibration_1std', 0)*100:.0f}%" if "calibration_1std" in r else "---"
    print(f"  {key:40s} | MAE={r['mae']:.2f} | Biais={r['bias']:+.2f} | 1σ={cal}")

## 8. Phase 2ter — Spatial MoE (Mixture of Experts, 20 features)

In [ ]:
import spatial_moe_model
from spatial_features import prepare_sessions_spatial

# Monkey-patch SpatialMoETrainer.fit
spatial_moe_model.SpatialMoETrainer.fit = _make_fast_fit(spatial_moe_model.SpatialMoETrainer.fit)

print("Préparation des sessions spatiales (20 features)...")
sessions_spatial = prepare_sessions_spatial(alerts)
print(f"  {len(sessions_spatial)} sessions, dim={sessions_spatial[0]['features'].shape[1]}")

result_moe, trainer_moe = spatial_moe_model.run_moe_experiment(sessions_spatial)

if result_moe:
    print(f"\nSpatial MoE → MAE={result_moe['mae']:.2f} | Biais={result_moe['bias']:+.2f} | P90={result_moe.get('p90', 0):.2f}")

## 9. Évaluation risque / gain (θ = 0.4)

Pour chaque modèle, on calcule à θ = 0.4 :
- **Gain** = Σ (last_lightning + 30 min − predicted_end) sur les alertes couvertes
- **Risque** = éclairs < 3 km après predicted_end / total éclairs < 3 km (cible < 2 %)

*Note : avec n_epochs=5 les métriques sont indicatives — smoke test uniquement.*

In [ ]:
import risk_evaluation as re_eval

THETA = 0.4
risk_results = {}

# ── Baseline 30 min ───────────────────────────────────────────────────────────
preds_baseline = re_eval.predictions_from_tabular(val_tab, "pred_baseline")
risk_results["Baseline 30 min"] = re_eval.evaluate_at_theta(preds_baseline, alerts, theta=THETA)

# ── XGBoost AFT ──────────────────────────────────────────────────────────────
preds_xgb = re_eval.predictions_from_tabular(val_tab, "pred_xgb_aft")
risk_results["XGBoost AFT"] = re_eval.evaluate_at_theta(preds_xgb, alerts, theta=THETA)

# ── BNN MC Dropout ───────────────────────────────────────────────────────────
val_bnn_with_pred = val_bnn.copy()
val_bnn_with_pred["pred_bnn"] = y_pred_bnn
preds_bnn = re_eval.predictions_from_tabular(val_bnn_with_pred, "pred_bnn")
risk_results["BNN MC Dropout"] = re_eval.evaluate_at_theta(preds_bnn, alerts, theta=THETA)

# ── Neural Hawkes V2 ─────────────────────────────────────────────────────────
for key, r in results_v2.items():
    if r is None or r.get("errors_df") is None:
        continue
    preds = re_eval.predictions_from_errors_df(r["errors_df"], alerts)
    risk_results[f"V2 {key}"] = re_eval.evaluate_at_theta(preds, alerts, theta=THETA)

# ── Hawkes Bayésien ──────────────────────────────────────────────────────────
for key, r in results_bayes.items():
    if r is None or r.get("errors_df") is None:
        continue
    use_unc = "uncertainty" in r["errors_df"].columns
    preds = re_eval.predictions_from_errors_df(r["errors_df"], alerts, use_uncertainty=use_unc)
    risk_results[f"Bayes {key}"] = re_eval.evaluate_at_theta(preds, alerts, theta=THETA)

# ── Spatial MoE ──────────────────────────────────────────────────────────────
if result_moe and result_moe.get("errors_df") is not None:
    use_unc = "uncertainty" in result_moe["errors_df"].columns
    preds = re_eval.predictions_from_errors_df(result_moe["errors_df"], alerts, use_uncertainty=use_unc)
    risk_results["Spatial MoE"] = re_eval.evaluate_at_theta(preds, alerts, theta=THETA)

re_eval.print_risk_table(risk_results, theta=THETA)


## 9. Résumé comparatif

In [ ]:
import numpy as np
import pandas as pd

rows = []

rows.append({"Phase": "1", "Modèle": "Baseline 30 min",
             "MAE": baseline_metrics["mae"], "RMSE": baseline_metrics["rmse"],
             "Biais": baseline_metrics["bias"]})
rows.append({"Phase": "1", "Modèle": "XGBoost survival:AFT",
             "MAE": xgb_metrics["mae"], "RMSE": xgb_metrics["rmse"],
             "Biais": xgb_metrics["bias"]})

bnn_mae  = float(np.mean(np.abs(y_true_bnn - y_pred_bnn)))
bnn_rmse = float(np.sqrt(np.mean((y_true_bnn - y_pred_bnn)**2)))
bnn_bias = float(np.mean(y_pred_bnn - y_true_bnn))
rows.append({"Phase": "1", "Modèle": "BNN MC Dropout (MLP)",
             "MAE": bnn_mae, "RMSE": bnn_rmse, "Biais": bnn_bias})

for key, r in results_v2.items():
    rows.append({"Phase": "2bis", "Modèle": f"V2 {key}",
                 "MAE": r["mae"], "RMSE": r.get("rmse", float("nan")),
                 "Biais": r["bias"]})

# V3 ignoré en local
for key in ["V3a GRU", "V3b TF", "V3c TF-Large", "V3d Ensemble"]:
    if key in results_v3:
        r = results_v3[key]
        rows.append({"Phase": "2bis V3", "Modèle": key,
                     "MAE": r["mae"], "RMSE": r.get("rmse", float("nan")),
                     "Biais": r["bias"]})

for key, r in results_bayes.items():
    rows.append({"Phase": "2bis", "Modèle": f"Bayes {key}",
                 "MAE": r["mae"], "RMSE": r.get("rmse", float("nan")),
                 "Biais": r["bias"]})

if result_moe:
    rows.append({"Phase": "2ter", "Modèle": "Spatial MoE",
                 "MAE": result_moe["mae"], "RMSE": result_moe.get("rmse", float("nan")),
                 "Biais": result_moe["bias"]})

summary_df = pd.DataFrame(rows).set_index(["Phase", "Modèle"]).round(2).sort_values("MAE")

print("\n" + "="*65)
print("  RÉSUMÉ COMPARATIF — toutes phases (local, n_epochs=" + str(N_EPOCHS_LOCAL) + ")")
print("="*65)
print(summary_df.to_string())

summary_df.to_csv(OUTPUT_DIR / "results_summary_local.csv")
print(f"\nTableau exporté : {OUTPUT_DIR / 'results_summary_local.csv'}")